### Step 1 — Import Libraries

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

### Step 2 — Load Dataset

In [3]:
df = pd.read_csv("../Dataset/f1_final_cleaned_data.csv")

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (7109, 28)


,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId,year,podium,driver_name,constructor_name,round,race_name,circuitId,date,circuit_name,location,country,qualifying_position
0,20323,337,4,6,8,3.0,1.0,1,1,25.0,49,45.0,1.0,1:58.287,191.706,1,2010,1,Fernando Alonso,Ferrari,1,Bahrain Grand Prix,3,2010-03-14,Bahrain International Circuit,Sakhir,Bahrain,3.0
1,20324,337,13,6,7,2.0,2.0,2,2,18.0,49,38.0,5.0,1:59.732,189.392,1,2010,1,Felipe Massa,Ferrari,1,Bahrain Grand Prix,3,2010-03-14,Bahrain International Circuit,Sakhir,Bahrain,2.0
2,20325,337,1,1,2,4.0,3.0,3,3,15.0,49,42.0,4.0,1:59.560,189.665,1,2010,1,Lewis Hamilton,McLaren,1,Bahrain Grand Prix,3,2010-03-14,Bahrain International Circuit,Sakhir,Bahrain,4.0
3,20326,337,20,9,5,1.0,4.0,4,4,12.0,49,32.0,12.0,2:00.218,188.627,1,2010,0,Sebastian Vettel,Red Bull,1,Bahrain Grand Prix,3,2010-03-14,Bahrain International Circuit,Sakhir,Bahrain,1.0
4,20327,337,3,131,4,5.0,5.0,5,5,10.0,49,45.0,13.0,2:00.236,188.599,1,2010,0,Nico Rosberg,Mercedes,1,Bahrain Grand Prix,3,2010-03-14,Bahrain International Circuit,Sakhir,Bahrain,5.0


### Step 3 — Check Columns

In [5]:
df.columns.tolist()

['resultId',
 'raceId',
 'driverId',
 'constructorId',
 'number',
 'grid',
 'position',
 'positionText',
 'positionOrder',
 'points',
 'laps',
 'fastestLap',
 'rank',
 'fastestLapTime',
 'fastestLapSpeed',
 'statusId',
 'year',
 'podium',
 'driver_name',
 'constructor_name',
 'round',
 'race_name',
 'circuitId',
 'date',
 'circuit_name',
 'location',
 'country',
 'qualifying_position']

### Step 4 — Create Driver Experience Feature

In [6]:
df["driver_experience"] = (
    df.groupby("driverId")["year"]
    .transform(lambda x: x - x.min())
)

#### Step 5 — Create Constructor Experience Feature

In [7]:
df["constructor_experience"] = (
    df.groupby("constructorId")["year"]
    .transform(lambda x: x - x.min())
)

### Step 6 — Create Grid Advantage Feature

In [9]:
df["grid_advantage"] = (
    df["grid"] - df["positionOrder"]
)

### Step 7 — Create Qualifying Advantage Feature

In [10]:
df["qualifying_advantage"] = (
    df["qualifying_position"] - df["positionOrder"]
)

### Step 8 — Create Driver Average Points Feature

In [11]:
df["driver_avg_points"] = (
    df.groupby("driverId")["points"]
    .transform("mean")
)

### Step 9 — Create Driver Podium Rate Feature

In [12]:
df["driver_podium_rate"] = (
    df.groupby("driverId")["podium"]
    .transform("mean")
)

### Step 10 — Create Constructor Podium Rate Feature

In [14]:
df["constructor_podium_rate"] = (
    df.groupby("constructorId")["podium"]
    .transform("mean")
)

### Step 11 — Create Recent Driver Performance

In [15]:
df = df.sort_values(
    ["driverId", "year"]
)

df["recent_driver_points"] = (
    df.groupby("driverId")["points"]
    .transform(
        lambda x: x.rolling(5, min_periods=1).mean()
    )
)

### Step 12 — Create Recent Constructor Performance

In [16]:
df = df.sort_values(
    ["constructorId", "year"]
)

df["recent_constructor_points"] = (
    df.groupby("constructorId")["points"]
    .transform(
        lambda x: x.rolling(5, min_periods=1).mean()
    )
)

### Step 13 — Define Target Variable

In [18]:
df["target_podium"] = df["podium"]

#### Step 14 — Select Features

In [19]:
features = [
    "year",
    "round",
    "driverId",
    "constructorId",
    "circuitId",
    "grid",
    "qualifying_position",
    "driver_experience",
    "constructor_experience",
    "grid_advantage",
    "qualifying_advantage",
    "driver_avg_points",
    "driver_podium_rate",
    "constructor_podium_rate",
    "recent_driver_points",
    "recent_constructor_points"
]

X = df[features]

y = df["target_podium"]

### Step 15 — Check Feature Dataset

In [20]:
print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

X.head()

Features Shape: (7109, 16)
Target Shape: (7109,)


,year,round,driverId,constructorId,circuitId,grid,qualifying_position,driver_experience,constructor_experience,grid_advantage,qualifying_advantage,driver_avg_points,driver_podium_rate,constructor_podium_rate,recent_driver_points,recent_constructor_points
2,2010,1,1,1,3,4.0,4.0,0,0,1.0,1.0,14.350592,0.532544,0.186667,15.000000,15.000000
29,2010,2,1,1,1,11.0,11.0,0,0,5.0,5.0,14.350592,0.532544,0.186667,11.500000,11.500000
53,2010,3,1,1,2,20.0,20.0,0,0,14.0,14.0,14.350592,0.532544,0.186667,10.333333,10.333333
73,2010,4,1,1,17,6.0,6.0,0,0,4.0,4.0,14.350592,0.532544,0.186667,12.250000,12.250000
109,2010,5,1,1,4,3.0,3.0,0,0,-11.0,-11.0,14.350592,0.532544,0.186667,9.800000,9.800000


### Step 16 — Save Feature Engineered Dataset

In [21]:
feature_data = pd.concat(
    [X, y],
    axis=1
)

feature_data.to_csv(
    "../Dataset/f1_feature_engineered_data.csv",
    index=False
)

print("Feature dataset saved successfully!")

Feature dataset saved successfully!
